In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Rétroaction classique et flux de contrôle (circuits dynamiques)

<Accordion>
<AccordionItem title="Package versions">

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
Les circuits dynamiques sont des outils puissants qui permettent de mesurer des qubits au milieu de l'exécution d'un circuit quantique, puis d'effectuer des opérations logiques classiques au sein du circuit en fonction du résultat de ces mesures intermédiaires. Ce processus est également appelé _rétroaction classique_. Bien que nous en soyons encore aux premières étapes de la compréhension de la meilleure façon de tirer parti des circuits dynamiques, la communauté de recherche quantique a déjà identifié plusieurs cas d'usage, notamment les suivants :

* Préparation efficace d'états quantiques, comme l'[état GHZ](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), l'[état W](https://arxiv.org/abs/2403.07604) (pour en savoir plus sur l'état W, voir aussi [« State preparation by shallow circuits using feed forward »](https://arxiv.org/abs/2307.14840)), et une large classe d'[états produits matriciels](https://arxiv.org/abs/2404.16083)
* [Intrication longue portée efficace](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) entre qubits d'une même puce en utilisant des circuits peu profonds
* [Échantillonnage efficace de circuits de type IQP](https://arxiv.org/pdf/2505.04705)
## Instruction `if`
L'instruction `if` permet d'effectuer des opérations de manière conditionnelle en fonction de la valeur d'un bit ou d'un registre classique.

Dans l'exemple ci-dessous, on applique une porte Hadamard à un qubit et on le mesure. Si le résultat est 1, on applique une porte X sur le qubit, ce qui a pour effet de le faire revenir à l'état 0. On mesure ensuite le qubit à nouveau. Le résultat de la mesure devrait être 0 avec une probabilité de 100 %.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

The `with` statement can be given an assignment target which is itself a context manager that can be stored and subsequently used to create an else block, which is executed whenever the contents of the `if` block are *not* executed.

In the example below, we initialize registers with two qubits and two classical bits. We apply a Hadamard gate to the first qubit and measure it. If the result is 1, then we apply a Hadamard gate on the second qubit; otherwise, we apply an X gate on the second qubit. Finally, we measure the second qubit as well.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

L'instruction `with` peut recevoir une cible d'affectation qui est elle-même un gestionnaire de contexte pouvant être sauvegardé et utilisé ultérieurement pour créer un bloc `else`, exécuté à chaque fois que le contenu du bloc `if` n'est *pas* exécuté.

Dans l'exemple ci-dessous, on initialise des registres avec deux qubits et deux bits classiques. On applique une porte Hadamard au premier qubit et on le mesure. Si le résultat est 1, on applique une porte Hadamard sur le second qubit ; sinon, on lui applique une porte X. Enfin, on mesure également le second qubit.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

En plus de conditionner sur un seul bit classique, il est également possible de conditionner sur la valeur d'un registre classique composé de plusieurs bits.

Dans l'exemple ci-dessous, on applique des portes Hadamard à deux qubits et on les mesure. Si le résultat est `01`, c'est-à-dire que le premier qubit est 1 et le second est 0, on applique une porte X à un troisième qubit. Enfin, on mesure le troisième qubit. À noter que par souci de clarté, nous avons choisi de préciser l'état du troisième bit classique, qui est 0, dans la condition `if`. Dans le dessin du circuit, la condition est indiquée par des cercles sur les bits classiques testés. Un cercle plein indique un conditionnement sur 1, tandis qu'un cercle vide indique un conditionnement sur 0.

In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

<span id="store"></span>
### `Store`

Tu peux utiliser l'instruction [`store`](https://docs.quantum.ibm.com/api/qiskit/circuit#store) pour enregistrer le résultat d'une expression classique, si cette expression est utilisée de manière répétée. Les opérations sont automatiquement parallélisées, ce qui rend ton code nettement plus efficace à l'exécution.

Par exemple, il est plus naturel et plus efficace à l'exécution d'écrire $B[0] \oplus B[1] \oplus B[2] \ldots$, où $B = \neg A$, que $(\neg A[0]) \oplus (\neg A[1]) \oplus (\neg A[2]) \ldots$.  La première forme calcule la négation en une seule étape parallèle avant la chaîne XOR, au lieu d'évaluer chaque négation séquentiellement dans l'expression.

Exemple complet :

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

qregs = QuantumRegister(4, "q")
creg = ClassicalRegister(3, "c")
# temp is a plain ClassicalRegister used as the store target
temp = ClassicalRegister(3, "temp")
qc = QuantumCircuit(qregs, creg, temp)

qc.h([0, 1, 2])
qc.measure([0, 1, 2], creg)

# Store bit-NOT of the full 3-bit register into temp
qc.store(temp, expr.bit_not(creg))

# Compute parity of temp using bit-indexed XOR
parity = expr.bit_xor(
    expr.bit_xor(expr.index(temp, 0), expr.index(temp, 1)),
    expr.index(temp, 2),
)

# Flip q3 if parity of ~creg is 1
with qc.if_test(parity):
    qc.x(3)

qc.measure([0, 1, 2], creg)

qc.draw("mpl")

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/e64ec241-41e8-40f8-ab64-af236c6c7802-0.avif" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/e64ec241-41e8-40f8-ab64-af236c6c7802-0.avif)

## Prochaines étapes
> **Tip:** - Apprends à implémenter un découplage dynamique précis en utilisant [stretch](/guides/stretch).
> - Utilise la [visualisation du calendrier du circuit](/guides/qiskit-runtime-circuit-timing) pour déboguer et optimiser tes circuits dynamiques.
> - [Exécute des circuits dynamiques](/guides/execute-dynamic-circuits).